데이터 다운로드 및 분석 

In [1]:
# Pytorch 라이브러리 로드
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
!pip install torchinfo
from torchinfo import summary
# Helper libraries
import numpy as np
import matplotlib.pyplot as plt



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#데이터 다운 및 로드 규칙 설정

#트랜스폼으로 이미지 불러올때 마다 정규화 하는 규칙
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

#이미지 다운로드
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

#학습 또는 평가 시 로드되는 데이터
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

C:\Users\Hwang\AppData\Roaming\Python\Python312\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
# 데이터셋을 로드하고 기본 정보를 확인해 보세요.
image, label = trainset[0] #0번째 데이터를 이미지와 레이블로 나눠서 변수화
class_names = trainset.classes
print(f"Image shape: {image.shape}")
print(f"Label: {label}")
print(f"Number of classes: {len(trainset.classes)}")
print(f"class_names : {class_names}")

Image shape: torch.Size([3, 32, 32])
Label: 6
Number of classes: 10
class_names : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
# 데이터의 개수도 확인해 봅시다.
train_size = torch.tensor(len(trainset))
test_size = torch.tensor(len(testset))

print(f"Train dataset size: {train_size} (Shape: {train_size.shape})")
print(f"Test dataset size: {test_size} (Shape: {test_size.shape})")

Train dataset size: 50000 (Shape: torch.Size([]))
Test dataset size: 10000 (Shape: torch.Size([]))
(50000, 32, 32, 3)


In [ ]:
#데이터 세트 시각화

#텐서형태 데이터를 역정규화 후 np array로 변환
def imshow(img):
    img = img / 2 + 0.5 #역정규화 과정(결과값 = (원본 - 평균) / 표준편차 --> 원본=  결과값*0.5+0.5)
    npimg = img.numpy()
    return np.transpose(npimg, (1, 2, 0))#mathplot에 맞게 변수 순서 변경(채널을 맨 뒤로)


#매스플랏으로 시각화
def show_multiple_images(dataset, n_images=9):
    dataiter = iter(dataset)
    images, labels = next(dataiter)
    fig, axes = plt.subplots(3, 3, figsize=(6, 6))
    axes = axes.flatten()

    for i in range(n_images):
        ax = axes[i]
        img = imshow(images[i])
        ax.imshow(img)
        ax.set_title(f"Label: {trainset.classes[labels[i]]}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()


 # 학습 데이터셋에서 9개의 이미지를 시각화합니다.
show_multiple_images(trainloader)

# 테스트 데이터셋에서 9개의 이미지를 시각화합니다.
show_multiple_images(testloader)

VGG블럭 설계 및 모델 생성 기초

In [ ]:
# function for building VGG Block

def build_vgg_block(input_layer,        #말그대로 압력처맵
                    num_cnn=3,          #반복 횟수(콘브레이어 갯수)
                    channel=64,         #필터 갯수:64
                    block_num=1):       #1번블록
    # 입력 레이어
    x = input_layer  #들어온 피처맵을 x라 하겠다

    # CNN 레이어
    layers = [] #여기에 레이어 쌓기
    in_channels = x.size(1) #인풋레이어의 사이즈 중에 (N,C H , W) 중 1번인 C의 사이즈 가꼬옴
    for cnn_num in range(num_cnn): #3번반복(위에 변수에서 지정)
        layers.append(      #아래 합성곱 연산을 레이어에 쌓는다
            nn.Conv2d(
                in_channels=in_channels,        #인풋의 채널수(만약 인풋이 원본이었다면 3, 반복중엔 64)
                out_channels=channel,           #필터 갯수:64
                kernel_size=3,                  #33필터, 스트1 , 패딩 1 -->크기 유지
                stride=1,                       
                padding=1,
            )
        )
        layers.append(nn.ReLU(inplace=True))        #렐루층을 추가
        in_channels = channel                      #맨처음 인채널이 3이었을 수도 있으니까 다음 연산부턴 인채널도 64로 맞춰줌)

    # Max Pooling 레이어 : 블록의 마무리. 풀링층 추가.
    layers.append(
        nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )
    )                       #풀링해서 크기 줄이기

    # Sequential으로 레이어 묶기 : 쌓인 레이어를 패키지해서 하나의 블럭으로 출력
    block = nn.Sequential(*layers)
    return block

In [16]:
#위에 설계한 블록으로 실제 모델 만들기
class VGGNet(nn.Module):    #모델의 이름 nn.Module: 기본틀
    def __init__(self):     #모델의 설정단계
        super(VGGNet, self).__init__() #모델생성시 국룰로 넣는 문구. nn모듈 설정을 그대로 쓰겠다?

        # VGG 블록 생성
        self.vgg_block = build_vgg_block(torch.zeros(1, 3, 32, 32))    #모델 안에 위에서 만든 블럭을 불러옴. (1, 3, 32, 32)의 0으로 꽉찬 텐서를 하나 넣어줘서 인풋데이터에 맞게 블럭을 세팅함
    def forward(self, x): # 모델의 실행 단계
        return self.vgg_block(x) #x가 들어오면 그 x를 vgg블럭 통과 시킨 결과를 내놓음

In [ ]:
# 블록 1개짜리 model 생성
model = VGGNet()  
print(model)

dummy_input = torch.zeros(1, 3, 32, 32)
output = model(dummy_input)
print(output.shape)



VGGNet(
  (vgg_block): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
)
torch.Size([1, 64, 16, 16])


In [4]:
# VGG 모델 자체를 생성하는 클래스입니다.
class VGG(nn.Module):
    def __init__(self, num_cnn_list=[2, 2, 3, 3, 3], channel_list=[64, 128, 256, 512, 512], num_classes=10):    #VGG 16 구조(self, 각 층의 콘브레이어 갯수, 각층의 채널수, 분류 클래스 갯수)
        super(VGG, self).__init__()
        assert len(num_cnn_list) == len(channel_list), "num_cnn_list와 channel_list의 길이가 일치해야 합니다."  #두 리스트 둘다 각 층의 설정이므로 층의 갯수가 같아야됨

        layers = [] #빈 레이어 생성, 아마 최종 레이어는 (콘 렐 콘 렐 풀 콘 렐 콘 렐 풀 콘 렐 콘 렌 콘 렐 풀..... 풀링으로 층 구별하고 각 층의 콘 렐 세트는 2 2 3 3 3 )
        in_channels = 3 #인풋채널은 rgb3

        for num_cnn, out_channels in zip(num_cnn_list, channel_list):   #5층이니까 5번반복. 반복마다 리스트에서 하나씩 꺼내서 매칭 (예:1층은 콘브층2개에 필터 64개라는 인풋을 밑에 블록생성 함수에 넘겨줌)
            layers.append(self._make_vgg_block(in_channels, out_channels, num_cnn)) #밑에서 정의한 함수로 쌓기. 콘브층 갯수와 필터 갯수는 위에 설정한 리스트와 매칭)
            in_channels = out_channels  # 다음 블록의 입력 채널을 설정

        self.feature_extractor = nn.Sequential(*layers)     #시퀀셜로 펼치기
        self.classifier = nn.Sequential(        #마지막 FC레이어 설계
            nn.Flatten(),                       #일렬로 펼치기
            nn.Linear(512 * 1 * 1, 4096),       # 512*1*1=512개의 인풋이 4096개의 출력으로 나옴, 1*1의미:32 32 인풋 이미지가 풀링 5번 거쳐서 1*1로 바뀜. 즉 플랫튼 하면 512개의 변수 생성
            nn.ReLU(True),                      #1대1 매칭 FC레이어
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Linear(4096, num_classes)        #인풋으로 4096개 받고 최종 클래스별 확률 내놓기(소프트맥스 없이 logit값만. 이래야 크로스 엔트로피등으로 로스만들기 가능)
        )

    def _make_vgg_block(self, in_channels, out_channels, num_cnn):      #블럭 생성기. 위에서 모델에 기계를 들여놓을때 편하게 하기 위한 함수.(self, 블록의 인풋 채널수, 아우풋채널수(필터갯수), 콘브층 갯수)
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1), nn.ReLU(True)]    #기본 레이어. 첫변수를 인채널 으로 지정해서 레이어를 시작. 처음 인채널은 3이고, 위쪽 포문에서 인채널=아웃채널 지정해줘서 최초 인채널은 이전 층의 채널수가 됨
        for _ in range(num_cnn - 1):    #기본레이어에 한번 있으니 -1. 리스트에 2 2 3 3 3 이니까 1 1 2 2 2번 반복
            layers.append(nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1))  #VGG국룰인 33필터에 패딩1 레이어 추가
            layers.append(nn.ReLU(True))
        layers.append(nn.MaxPool2d(kernel_size=2, stride=2))  # Max Pooling 추가
        return nn.Sequential(*layers)   #시퀀셜로 연결

    def forward(self, x):
        x = self.feature_extractor(x) #일단 x를 5층짜리 합성곱레이어 집단에 넣어줌
        x = self.classifier(x)      # 그 결과를 FC레이어로 보냄
        return x

그니까 맨 위에서 몇 층 구조인지 층마다 콘브레이어 수랑 채널(필터)갯수, 분류 클래스 갯수 지정해주고, 빈레이어 지정하고, 인채널3(rgb)만 지정한담에. 레이어 쌓는데, 하나하나 쌓기 귀찬으니까 일단 1층부터 5층까지는 포문으로 지정해서 근데 그 포문안에는 또 블럭생성 함수가 있고 블럭생성 함수는 그냥 단순히 콘브층 갯수랑 필터수 맞게레이어 쌓아주고 마지막에 풀링 층 넣어주는거고, 처음 포문에서 zip으로 자종햤우나까 마지막 5층 콘브3개->풀링 쌀고 레이어[]는 다 채워진거고. 포워드문 보면 x를 저 5층레이어집단 먼저 통과시키고 거기서 나온걸다시 fc레이어 시퀀셜된거에 통과시키는거고

In [5]:
# 기본값을 그대로 사용해서 VGG 모델을 만들면 VGG-16이 됩니다.

vgg_16 = VGG()
summary(vgg_16)

Layer (type:depth-idx)                   Param #
VGG                                      --
├─Sequential: 1-1                        --
│    └─Sequential: 2-1                   --
│    │    └─Conv2d: 3-1                  1,792
│    │    └─ReLU: 3-2                    --
│    │    └─Conv2d: 3-3                  36,928
│    │    └─ReLU: 3-4                    --
│    │    └─MaxPool2d: 3-5               --
│    └─Sequential: 2-2                   --
│    │    └─Conv2d: 3-6                  73,856
│    │    └─ReLU: 3-7                    --
│    │    └─Conv2d: 3-8                  147,584
│    │    └─ReLU: 3-9                    --
│    │    └─MaxPool2d: 3-10              --
│    └─Sequential: 2-3                   --
│    │    └─Conv2d: 3-11                 295,168
│    │    └─ReLU: 3-12                   --
│    │    └─Conv2d: 3-13                 590,080
│    │    └─ReLU: 3-14                   --
│    │    └─Conv2d: 3-15                 590,080
│    │    └─ReLU: 3-16                  

In [6]:
# 원하는 블록의 설계에 따라 매개변수로 리스트를 전달해 줍니다.
vgg_19 = VGG(
    num_cnn_list=[2, 2, 4, 4, 4],
    channel_list=[64, 128, 256, 512, 512]
)
summary(vgg_19)

Layer (type:depth-idx)                   Param #
VGG                                      --
├─Sequential: 1-1                        --
│    └─Sequential: 2-1                   --
│    │    └─Conv2d: 3-1                  1,792
│    │    └─ReLU: 3-2                    --
│    │    └─Conv2d: 3-3                  36,928
│    │    └─ReLU: 3-4                    --
│    │    └─MaxPool2d: 3-5               --
│    └─Sequential: 2-2                   --
│    │    └─Conv2d: 3-6                  73,856
│    │    └─ReLU: 3-7                    --
│    │    └─Conv2d: 3-8                  147,584
│    │    └─ReLU: 3-9                    --
│    │    └─MaxPool2d: 3-10              --
│    └─Sequential: 2-3                   --
│    │    └─Conv2d: 3-11                 295,168
│    │    └─ReLU: 3-12                   --
│    │    └─Conv2d: 3-13                 590,080
│    │    └─ReLU: 3-14                   --
│    │    └─Conv2d: 3-15                 590,080
│    │    └─ReLU: 3-16                  

In [7]:
# 원하는 블록의 설계에 따라 매개변수로 리스트를 전달해 줍니다.
vgg_13 = VGG(
    num_cnn_list=[2, 2, 2, 2, 2],
    channel_list=[64, 128, 256, 512, 512]
)
summary(vgg_13)

Layer (type:depth-idx)                   Param #
VGG                                      --
├─Sequential: 1-1                        --
│    └─Sequential: 2-1                   --
│    │    └─Conv2d: 3-1                  1,792
│    │    └─ReLU: 3-2                    --
│    │    └─Conv2d: 3-3                  36,928
│    │    └─ReLU: 3-4                    --
│    │    └─MaxPool2d: 3-5               --
│    └─Sequential: 2-2                   --
│    │    └─Conv2d: 3-6                  73,856
│    │    └─ReLU: 3-7                    --
│    │    └─Conv2d: 3-8                  147,584
│    │    └─ReLU: 3-9                    --
│    │    └─MaxPool2d: 3-10              --
│    └─Sequential: 2-3                   --
│    │    └─Conv2d: 3-11                 295,168
│    │    └─ReLU: 3-12                   --
│    │    └─Conv2d: 3-13                 590,080
│    │    └─ReLU: 3-14                   --
│    │    └─MaxPool2d: 3-15              --
│    └─Sequential: 2-4                   --
│

In [ ]:
BATCH_SIZE = 256
EPOCH = 15

In [ ]:
# CIFAR-10 데이터셋에 대해 Normalize와 Tensor 변환을 적용하는 코드
transform = transforms.Compose([
    transforms.ToTensor(),  # 이미지를 Tensor로 변환
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # VGG-16 표준 정규화
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
import time

current_time = time.time()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vgg_16 = torchvision.models.vgg16(pretrained=True)
vgg_16.to(device)

for param in vgg_16.parameters():
    param.requires_grad = True
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(vgg_16.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)

vgg_16_train_losses = []
vgg_16_val_accuracy = []

for epoch in range(EPOCH):
    vgg_16.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (inputs, labels) in enumerate(trainloader, 0):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = vgg_16(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if i % 100 == 99:
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / i + 1:.3f}")
            
    train_loss = running_loss / len(trainloader)
    train_acc = 100 * correct / total
    vgg_16_train_losses.append(train_loss)

    print(f"Epoch {epoch + 1}: Train Accuracy: {train_acc:.2f}%")

    vgg_16.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = vgg_16(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    vgg_16_val_accuracy.append(val_acc)

    print(f"Epoch {epoch + 1}: Validation Accuracy: {val_acc:.2f}%")

print("Finished Training")
print(time.time() - current_time)

In [ ]:
import time  # 시간 측정을 위한 라이브러리 로드

current_time = time.time()  # 학습이 시작되는 현재 시각 기록(타이머 시작)

# 계산을 수행할 장비(GPU 또는 CPU) 선택
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 토치비전에서 이미 학습이 완료된(pretrained=True) vgg16 모델을 가져옴
vgg_16 = torchvision.models.vgg16(pretrained=True)
vgg_16.to(device)  # 모델의 모든 가중치를 선택한 장비(GPU 등)로 전송

# 모델의 모든 파라미터가 학습 과정에서 업데이트되도록 설정 (가중치 수정 허용)
for param in vgg_16.parameters():
    param.requires_grad = True

# 손실 함수 설정: 다중 클래스 분류를 위한 크로스 엔트로피 사용 (내부에 소프트맥스 포함)
criterion = nn.CrossEntropyLoss()
# 최적화 알고리즘: 확률적 경사 하강법(SGD) 사용. 모델의 가중치를 어떻게 수정할지 결정
optimizer = optim.SGD(vgg_16.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)

# 학습 과정의 손실 값과 정확도를 저장하기 위한 빈 리스트 생성
vgg_16_train_losses = []
vgg_16_val_accuracy = []

# 정해진 에포크(전체 데이터를 몇 번 반복해서 볼지)만큼 반복
for epoch in range(EPOCH):
    vgg_16.train()  # 모델을 '학습 모드'로 설정 (드롭아웃 등이 작동함)
    running_loss = 0.0  # 이번 에포크의 누적 손실 초기화
    correct = 0  # 맞춘 개수 초기화
    total = 0  # 전체 데이터 개수 초기화

    # 학습 데이터로더에서 배치(묶음) 단위로 데이터를 꺼내옴
    for i, (inputs, labels) in enumerate(trainloader, 0):
        # 데이터와 정답을 사용할 장비(GPU 등)로 전송
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()  # 지난번 계산했던 기울기(미분값)를 0으로 초기화
        outputs = vgg_16(inputs)  # 모델에 데이터를 넣어 예측값(출력)을 얻음 (Forward)
        loss = criterion(outputs, labels)  # 예측값과 실제 정답을 비교해 틀린 정도(Loss) 계산
        loss.backward()  # 틀린 정도를 바탕으로 각 가중치가 얼마나 변해야 할지 계산 (Backpropagation)
        optimizer.step()  # 계산된 결과를 바탕으로 실제 모델의 가중치를 한 단계 업데이트

        running_loss += loss.item()  # 현재 배치의 손실 값을 누적
        _, predicted = torch.max(outputs, 1)  # 예측값 중 가장 높은 점수를 가진 클래스 선택
        total += labels.size(0)  # 처리한 총 데이터 개수 누적
        correct += (predicted == labels).sum().item()  # 정답과 일치하는 개수 누적

        # 100번째 배치마다 현재까지의 평균 손실 출력 (중간 점검)
        if i % 100 == 99:
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.3f}")
            
    # 이번 에포크의 평균 손실과 정확도 계산 및 저장
    train_loss = running_loss / len(trainloader)
    train_acc = 100 * correct / total
    vgg_16_train_losses.append(train_loss)

    print(f"Epoch {epoch + 1}: Train Accuracy: {train_acc:.2f}%")

    # --- 검증 단계 (Validation) ---
    vgg_16.eval()  # 모델을 '평가 모드'로 설정 (학습 중단, 드롭아웃 꺼짐)
    correct = 0
    total = 0

    # 평가할 때는 가중치를 수정할 필요가 없으므로 미분 계산을 하지 않음 (메모리 절약)
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = vgg_16(inputs)  # 예측값 얻기
            _, predicted = torch.max(outputs, 1) # 가장 높은 확률의 클래스 선택
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    # 검증 데이터셋에 대한 정확도 계산 및 저장
    val_acc = 100 * correct / total
    vgg_16_val_accuracy.append(val_acc)

    print(f"Epoch {epoch + 1}: Validation Accuracy: {val_acc:.2f}%")

print("Finished Training")  # 모든 학습 완료 메시지
print(f"Total training time: {time.time() - current_time:.2f} seconds")  # 총 걸린 시간 출력


결국 코드 복잡한데 로직은 이론대로 순전파 역전파 파라미터 수정이고, 이때 평가 바구니 채우기 위해 배치마다 손실값이랑 맞춘갯수 누적해서 1에포크완료했을때 전체 손실을 트레인로더 숫자로 나눈걸 로스 바구니에 넣고 트레이닝 셋으로 순전파해서 나온 정확도를 정확도 바구니에 넣는거구나